<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026/%D0%93%D0%BB%D0%B0%D0%B2%D0%B0_6_%D0%91%D0%B0%D0%B9%D0%B5%D1%81%D0%BE%D0%B2%D1%81%D0%BA%D0%B8%D0%B5_%D0%BC%D0%B5%D1%82%D0%BE%D0%B4%D1%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Глава 6. Байесовские методы  
## Часть I. Введение в байесовскую парадигму и наивный байесовский классификатор (теория)

В классическом машинном обучении мы искали точечные оценки параметров — например, методом максимального правдоподобия. Но что, если мы хотим не просто «лучшую» оценку, а полное распределение возможных значений параметров, учитывающее нашу априорную уверенность и ограниченность данных? Байесовский подход даёт именно это — он трактует параметры модели как случайные величины и обновляет наши представления о них по мере поступления данных.

### 1. Байесовская парадигма

В основе байесовского подхода лежит вероятностная модель, описывающая совместное распределение наблюдаемых данных $\mathcal{D}$ и неизвестных параметров модели $\boldsymbol{\theta}$:
$$
p(\mathcal{D}, \boldsymbol{\theta}) = p(\mathcal{D} \mid \boldsymbol{\theta})\,p(\boldsymbol{\theta}).
$$
Здесь $p(\boldsymbol{\theta})$ — **априорное распределение**, отражающее наши знания или убеждения о параметрах до наблюдения данных, а $p(\mathcal{D} \mid \boldsymbol{\theta})$ — **функция правдоподобия**, описывающая вероятность получения данных при фиксированном значении параметров.

Наблюдая данные, мы пересчитываем распределение параметров с помощью **теоремы Байеса**:
$$
p(\boldsymbol{\theta} \mid \mathcal{D}) = \frac{p(\mathcal{D} \mid \boldsymbol{\theta})\,p(\boldsymbol{\theta})}{p(\mathcal{D})}, \qquad
p(\mathcal{D}) = \int p(\mathcal{D} \mid \boldsymbol{\theta})\,p(\boldsymbol{\theta})\,d\boldsymbol{\theta}.
\tag{1.1}
$$
Величина $p(\boldsymbol{\theta} \mid \mathcal{D})$ называется **апостериорным распределением** — оно объединяет априорную информацию и информацию, привнесённую данными. Нормировочная константа $p(\mathcal{D})$ носит название **маргинального правдоподобия** (или evidence) и играет ключевую роль при сравнении моделей: чем сложнее модель, тем сильнее она «размазывает» вероятность по пространству данных, что штрафуется в $p(\mathcal{D})$.

Получив апостериорное распределение, можно делать предсказания для нового наблюдения $\mathbf{x}_*$, не фиксируя одно «наилучшее» значение параметров. **Предсказательное распределение** вычисляется усреднением по всем возможным значениям параметров, взвешенным их апостериорной вероятностью:
$$
p(y_* \mid \mathbf{x}_*, \mathcal{D}) = \int p(y_* \mid \mathbf{x}_*, \boldsymbol{\theta})\,p(\boldsymbol{\theta} \mid \mathcal{D})\,d\boldsymbol{\theta}.
\tag{1.2}
$$
Такой подход называется **байесовским усреднением моделей** (Bayesian model averaging). Его принципиальное отличие от частотной парадигмы — вместо точечных оценок мы работаем с распределениями, что позволяет естественно учитывать неопределённость, связанную с конечностью выборки.

### 2. Наивный байесовский классификатор: вероятностная модель

Рассмотрим задачу классификации: по вектору признаков $\mathbf{x} = (x_1, \dots, x_D)$ требуется отнести объект к одному из классов $C_k$, $k = 1,\dots,K$. Идеальный классификатор, минимизирующий вероятность ошибки, выбирает класс с максимальной апостериорной вероятностью $P(C_k \mid \mathbf{x})$. По теореме Байеса
$$
P(C_k \mid \mathbf{x}) = \frac{P(C_k)\,P(\mathbf{x} \mid C_k)}{P(\mathbf{x})} \propto P(C_k)\,P(\mathbf{x} \mid C_k).
\tag{2.1}
$$
Знаменатель $P(\mathbf{x})$ не зависит от класса, поэтому для принятия решения достаточно сравнить числители.

Прямая оценка многомерного условного распределения $P(\mathbf{x} \mid C_k)$ требует огромного объёма данных. **Наивный байесовский классификатор** (Naive Bayes) обходит эту трудность с помощью сильного упрощающего предположения: **признаки условно независимы при фиксированном классе**. Формально,
$$
P(\mathbf{x} \mid C_k) = \prod_{j=1}^D P(x_j \mid C_k).
\tag{2.2}
$$
Подставляя (2.2) в (2.1), получаем решающее правило
$$
P(C_k \mid \mathbf{x}) \propto P(C_k) \prod_{j=1}^D P(x_j \mid C_k).
\tag{2.3}
$$
Хотя предположение о независимости на практике почти всегда нарушается, оно резко сокращает число оцениваемых параметров: вместо полного совместного распределения с экспоненциальным (по $D$) числом параметров мы оцениваем лишь одномерные условные распределения для каждого признака. Это делает наивный байесовский классификатор вычислительно эффективным и позволяет ему работать даже при очень высокой размерности признакового пространства.

### 3. Оценивание параметров

Для использования правила (2.3) необходимо по обучающей выборке оценить априорные вероятности классов $P(C_k)$ и условные распределения признаков $P(x_j \mid C_k)$. Оценки обычно получают методом максимального правдоподобия (MLE) или их регуляризованными вариантами.

**Априорные вероятности классов.** Если нет дополнительной информации, их оценивают как долю объектов класса $k$ в выборке:
$$
P(C_k) = \frac{N_k}{n},
$$
где $N_k$ — число объектов класса $k$, $n$ — общий объём выборки.

**Дискретные признаки.** Пусть $j$-й признак принимает значения из конечного множества $\{1,\dots,V_j\}$. Максимально правдоподобная оценка условной вероятности значения $v$ равна
$$
P(x_j = v \mid C_k) = \frac{N_{kjv}}{N_k},
$$
где $N_{kjv}$ — количество объектов класса $k$, у которых признак $j$ принял значение $v$. Такая оценка страдает от проблемы нулевых частот: если какое-то значение ни разу не встретилось в обучающих данных, его вероятность приравнивается к нулю, что обнуляет всё произведение (2.3) и делает классификатор крайне неустойчивым.

Проблему решает **сглаживание Лапласа** (или аддитивное сглаживание):
$$
P(x_j = v \mid C_k) = \frac{N_{kjv} + \alpha}{N_k + \alpha V_j},
\tag{3.1}
$$
где $\alpha \ge 0$ — параметр сглаживания, $V_j$ — число возможных значений признака. При $\alpha = 1$ получаем классическое сглаживание Лапласа; при $\alpha = 0$ — MLE; при $0 < \alpha < 1$ говорят о сглаживании Лидстоуна. Параметр $\alpha$ интерпретируется как «псевдо‑счётчики»: мы как бы добавляем $\alpha$ фиктивных наблюдений каждого значения к обучающей выборке, что не позволяет вероятностям обращаться в ноль.

**Непрерывные признаки.** Для непрерывных признаков часто предполагают, что они в пределах каждого класса распределены нормально:
$$
P(x_j \mid C_k) = \frac{1}{\sqrt{2\pi\sigma_{jk}^2}} \exp\!\left(-\frac{(x_j - \mu_{jk})^2}{2\sigma_{jk}^2}\right).
$$
Параметры $\mu_{jk}$ и $\sigma_{jk}^2$ оценивают как выборочное среднее и выборочную дисперсию (с поправкой на смещение или без неё) по объектам класса $k$. Возможны и другие параметрические модели (например, мультиномиальное или бернуллиевское распределение), в зависимости от природы признаков.

> **Углублённый взгляд: байесовский вывод для мультиномиального распределения с априорным Дирихле**
>
> Сглаживание Лапласа можно получить строго в рамках байесовского подхода. Пусть для фиксированного класса $k$ и признака $j$ вероятности значений $\boldsymbol{\theta} = (\theta_1,\dots,\theta_V)$, $\sum_v \theta_v = 1$, имеют априорное распределение Дирихле:
> $$
> p(\boldsymbol{\theta}) = \frac{\Gamma(\alpha V)}{\Gamma(\alpha)^V} \prod_{v=1}^V \theta_v^{\alpha - 1},
> $$
> где $\alpha > 0$ — параметр концентрации. Правдоподобие наблюдаемых частот $\mathbf{N} = (N_1,\dots,N_V)$ мультиномиально:
> $$
> p(\mathbf{N} \mid \boldsymbol{\theta}) \propto \prod_{v=1}^V \theta_v^{N_v}.
> $$
> По теореме Байеса апостериорное распределение вновь является Дирихле с обновлёнными параметрами $\alpha_v = N_v + \alpha$. В качестве точечной оценки можно взять среднее апостериорного распределения:
> $$
> \mathbb{E}[\theta_v \mid \mathbf{N}] = \frac{N_v + \alpha}{\sum_{v'} (N_{v'} + \alpha)} = \frac{N_v + \alpha}{N + \alpha V},
> $$
> что в точности совпадает с формулой (3.1). Таким образом, сглаживание Лапласа есть не эвристика, а байесовская оценка с априорным распределением Дирихле, где параметр $\alpha$ задаёт силу априорного убеждения в равновероятности всех значений.

### 4. Решающее правило

Для классификации нового объекта $\mathbf{x}$ выбирается класс, максимизирующий апостериорную вероятность (MAP — maximum a posteriori):
$$
\hat{y} = \arg\max_k \; P(C_k \mid \mathbf{x}) = \arg\max_k \; \Bigl[ \log P(C_k) + \sum_{j=1}^D \log P(x_j \mid C_k) \Bigr].
\tag{4.1}
$$
Переход к логарифмам не меняет положения максимума, но делает вычисления численно устойчивыми: произведение большого числа вероятностей, близких к нулю, заменяется суммой логарифмов. На практике именно логарифмическая форма используется во всех реализациях.

### 5. Почему наивный байесовский классификатор работает даже при нарушении независимости?

Предположение об условной независимости признаков выглядит чрезмерно сильным, и в реальных данных оно редко выполняется. Тем не менее наивный байесовский классификатор часто демонстрирует высокое качество, сравнимое с более сложными моделями. Причина кроется в том, что для правильной классификации не обязательно точно оценивать апостериорные вероятности — достаточно правильно упорядочить классы по величине $P(C_k \mid \mathbf{x})$.

Рассмотрим ситуацию с двумя коррелированными признаками. Если оба признака дают информацию о классе, их корреляция часто «работает» в одну и ту же сторону для всех классов, и произведение $P(x_1 \mid C_k)P(x_2 \mid C_k)$ оказывается монотонной функцией от истинного совместного распределения $P(x_1, x_2 \mid C_k)$ при фиксации класса. Даже если численные значения вероятностей смещены, решающая граница, определяемая равенством апостериорных вероятностей классов, может оставаться близкой к оптимальной.

Кроме того, наивный байесовский классификатор обладает низкой дисперсией: он оценивает относительно мало параметров, что делает его устойчивым к переобучению на малых выборках. В условиях, когда данные ограничены, это преимущество часто перевешивает систематическую ошибку, вызванную нарушением независимости. Именно поэтому наивный байес до сих пор широко применяется в задачах текстовой классификации, фильтрации спама и других областях с высокой размерностью.

---

## Вопросы для самопроверки

### Для начинающих
1. Что даёт априорное распределение в байесовском подходе? Какую роль оно играет при переходе к апостериорному распределению?
2. В чём заключается «наивность» наивного байесовского классификатора? Какое предположение он делает о признаках?
3. Какие разновидности наивного байесовского классификатора вы знаете в зависимости от типа признаков?
4. Зачем применяется сглаживание Лапласа? К каким последствиям приводит его отсутствие?
5. Как вычисляется предсказание наивного байесовского классификатора для нового объекта? Зачем при этом переходят к логарифмам?

### Для профессионалов
1. Докажите, что сглаживание Лапласа (3.1) является байесовской оценкой (средним апостериорного распределения) при априорном распределении Дирихле. Как интерпретировать параметр $\alpha$?
2. Объясните, почему наивный байесовский классификатор часто даёт корректную решающую границу даже при сильной корреляции признаков. Приведите пример, когда его предположение о независимости фатально ухудшает качество.
3. Почему наивный байесовский классификатор работает очень быстро по сравнению, например, с SVM или нейронными сетями? Какие этапы обучения и предсказания определяют его вычислительную сложность?
4. Как выбрать оптимальное значение $\alpha$ в сглаживании Лапласа, не прибегая к перебору по валидационной выборке? Какие байесовские методы могут автоматически определить степень сглаживания?

```python
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import load_breast_cancer, fetch_20newsgroups
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
```


## Часть II. Реализация наивного байесовского классификатора и его применение

Теоретическая простота наивного байесовского классификатора позволяет реализовать его буквально в несколько строк, а его вычислительная эффективность делает его незаменимым инструментом для задач с высокой размерностью, особенно в текстовой классификации. В этой части мы напишем собственный Gaussian NB, применим Multinomial NB к реальным текстовым данным и сравним наивный байесовский подход с логистической регрессией.

### 1. Реализация Gaussian NB с нуля

Для непрерывных признаков гауссовский наивный байесовский классификатор предполагает, что каждый признак при фиксированном классе распределён нормально. Параметры распределения — среднее $\mu_{jk}$ и дисперсия $\sigma_{jk}^2$ — оцениваются по обучающей выборке для каждого класса $k$ и каждого признака $j$. Реализуем класс, который выполняет эти оценки и делает предсказания.

```python
class GaussianNaiveBayes:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        self.means_ = np.zeros((self.n_classes_, X.shape[1]))
        self.vars_ = np.zeros((self.n_classes_, X.shape[1]))
        self.priors_ = np.zeros(self.n_classes_)
        for idx, k in enumerate(self.classes_):
            X_k = X[y == k]
            self.means_[idx, :] = X_k.mean(axis=0)
            # MLE-дисперсия (деление на N_k), совпадает с np.var(ddof=0)
            self.vars_[idx, :] = np.var(X_k, axis=0, ddof=0)
            self.priors_[idx] = np.log(X_k.shape[0] / X.shape[0])
        return self

    def predict(self, X):
        log_likelihood = np.zeros((X.shape[0], self.n_classes_))
        for idx in range(self.n_classes_):
            # Логарифм гауссовской плотности для всех объектов сразу
            log_prob = -0.5 * np.log(2 * np.pi * self.vars_[idx, :])
            log_prob -= 0.5 * ((X - self.means_[idx, :]) ** 2) / self.vars_[idx, :]
            log_likelihood[:, idx] = log_prob.sum(axis=1) + self.priors_[idx]
        return self.classes_[np.argmax(log_likelihood, axis=1)]
```

Несколько замечаний по реализации.

- Дисперсия вычисляется с помощью `np.var(..., ddof=0)`, что соответствует оценке максимального правдоподобия (деление на $N_k$, а не на $N_k-1$). В библиотеке `sklearn.naive_bayes.GaussianNB` по умолчанию также используется MLE-дисперсия, но затем к ней добавляется небольшое сглаживающее слагаемое `var_smoothing` (по умолчанию $10^{-9}$), чтобы избежать вырожденных случаев при нулевой дисперсии. Для простоты мы опускаем это сглаживание, однако в реальных приложениях оно крайне рекомендуется.
- Для численной устойчивости мы работаем с логарифмами: логарифм гауссовской плотности раскладывается в сумму, а логарифм априорной вероятности класса прибавляется один раз. Это эквивалентно правилу (4.1) из предыдущей части, но реализовано с использованием матричных операций для всех тестовых объектов сразу.

Проверим нашу реализацию на наборе данных breast_cancer и сравним с библиотечным `GaussianNB`.

```python
# Загрузка и подготовка данных
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Наш классификатор
our_nb = GaussianNaiveBayes()
our_nb.fit(X_train, y_train)
our_pred = our_nb.predict(X_test)
our_acc = accuracy_score(y_test, our_pred)

# Библиотечный GaussianNB
sk_nb = GaussianNB()
sk_nb.fit(X_train, y_train)
sk_pred = sk_nb.predict(X_test)
sk_acc = accuracy_score(y_test, sk_pred)

print(f"Свой GaussianNB: accuracy = {our_acc:.4f}")
print(f"sklearn GaussianNB: accuracy = {sk_acc:.4f}")

# Матрицы ошибок
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, our_pred, ax=axes[0], colorbar=False)
axes[0].set_title("Свой GaussianNB")
ConfusionMatrixDisplay.from_predictions(y_test, sk_pred, ax=axes[1], colorbar=False)
axes[1].set_title("sklearn GaussianNB")
plt.tight_layout()
plt.show()
```

Результаты должны быть практически идентичны, за исключением возможного небольшого расхождения из-за отсутствия сглаживания дисперсии в нашей реализации (что в данном случае не критично, так как все признаки имеют ненулевую дисперсию).

### 2. Применение Multinomial NB к текстовым данным

Наивный байесовский классификатор исторически зарекомендовал себя как один из лучших методов для классификации текстов. В этой задаче признаки — это слова (или n-граммы), и предположение об условной независимости работает удивительно хорошо. Мультиномиальная модель `MultinomialNB` рассматривает каждый документ как последовательность событий «слово появилось», а правдоподобие класса моделируется мультиномиальным распределением.

Загрузим подмножество новостных групп 20 newsgroups, преобразуем тексты в мешок слов и обучим классификатор.

```python
# Выбираем четыре категории для наглядности
categories = ['rec.sport.baseball', 'sci.space',
              'talk.politics.mideast', 'comp.graphics']
news_train = fetch_20newsgroups(subset='train', categories=categories, random_state=42)
news_test = fetch_20newsgroups(subset='test', categories=categories, random_state=42)

# Мешок слов
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(news_train.data)
X_test_counts = vectorizer.transform(news_test.data)

print("Размерность признакового пространства:", X_train_counts.shape[1])
```

Обучим `MultinomialNB` со стандартным сглаживанием Лапласа ($\alpha=1.0$) и оценим качество.

```python
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_train_counts, news_train.target)
pred = mnb.predict(X_test_counts)
acc = accuracy_score(news_test.target, pred)
print(f"MultinomialNB (alpha=1.0): accuracy = {acc:.4f}")
print("\nClassification report:\n", classification_report(news_test.target, pred, target_names=news_train.target_names))
```

Построим матрицу ошибок.

```python
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(news_test.target, pred,
                                       display_labels=news_train.target_names, ax=ax, colorbar=False)
plt.title("Матрица ошибок MultinomialNB")
plt.show()
```

Теперь исследуем влияние параметра сглаживания `alpha`. Переберём несколько значений на логарифмической шкале и построим график точности на валидации (используем встроенную кросс‑валидацию через `GridSearchCV`, но для наглядности можно просто разбить тренировочную выборку или использовать `GridSearchCV` на 3‑х фолдах).

```python
alphas = np.logspace(-3, 3, 20)
params = {'alpha': alphas}
grid = GridSearchCV(MultinomialNB(), param_grid=params, cv=3, scoring='accuracy')
grid.fit(X_train_counts, news_train.target)

plt.figure(figsize=(8, 5))
plt.semilogx(alphas, grid.cv_results_['mean_test_score'], marker='o')
plt.xlabel('alpha')
plt.ylabel('CV accuracy')
plt.title('Влияние сглаживания на качество MultinomialNB')
plt.grid()
plt.show()
print(f"Оптимальное alpha: {grid.best_params_['alpha']:.4f}, лучшая CV accuracy: {grid.best_score_:.4f}")
```

График обычно демонстрирует, что слишком маленькие $\alpha$ ведут к переобучению, слишком большие — к излишнему сглаживанию и потере точности; оптимум часто лежит в районе $\alpha < 1$.

Одно из преимуществ наивного байесовского классификатора — его интерпретируемость. Мы можем извлечь наиболее характерные слова для каждого класса, посмотрев на логарифмы вероятностей `feature_log_prob_` (это $\log P(x_j \mid C_k)$).

```python
feature_names = np.array(vectorizer.get_feature_names_out())
for idx, class_name in enumerate(news_train.target_names):
    top_indices = np.argsort(mnb.feature_log_prob_[idx])[::-1][:10]
    top_words = feature_names[top_indices]
    print(f"\nТоп-10 слов для класса '{class_name}':")
    print(", ".join(top_words))
```

Слова чётко отражают тематику: например, для `sci.space` — «space», «nasa», «orbit»; для `rec.sport.baseball` — «game», «team», «baseball». Это делает модель не только точной, но и прозрачной.

### 3. Сравнение NB с логистической регрессией

Наивный байесовский классификатор и логистическая регрессия часто применяются к одним и тем же задачам, но исходят из принципиально разных предпосылок. NB — **генеративная модель**: она описывает, как порождаются данные $P(\mathbf{x} \mid y)$, и затем с помощью теоремы Байеса получает $P(y \mid \mathbf{x})$. Логистическая регрессия — **дискриминативная модель**: она напрямую моделирует $P(y \mid \mathbf{x})$, не пытаясь описать распределение признаков.

Сравним их на том же текстовом датасете.

```python
# Логистическая регрессия с параметрами по умолчанию
lr = LogisticRegression(max_iter=1000)
start = time.time()
lr.fit(X_train_counts, news_train.target)
lr_train_time = time.time() - start

start = time.time()
mnb.fit(X_train_counts, news_train.target)
nb_train_time = time.time() - start

lr_pred = lr.predict(X_test_counts)
mnb_pred = mnb.predict(X_test_counts)

lr_acc = accuracy_score(news_test.target, lr_pred)
mnb_acc = accuracy_score(news_test.target, mnb_pred)

print(f"MultinomialNB: accuracy={mnb_acc:.4f}, время обучения={nb_train_time:.4f}с")
print(f"LogisticRegression: accuracy={lr_acc:.4f}, время обучения={lr_train_time:.4f}с")
```

Как правило, логистическая регрессия показывает немного более высокую точность, особенно при большом количестве данных, поскольку она не связана наивным предположением о независимости и может улавливать взаимосвязи между словами. Однако NB обучается практически мгновенно, так как сводится к простому подсчёту частот, тогда как логистическая регрессия требует итеративной оптимизации. На малых выборках NB часто работает лучше благодаря низкой дисперсии, в то время как логистическая регрессия может переобучаться.

С точки зрения интерпретируемости оба метода прозрачны: NB позволяет непосредственно увидеть «любимые» слова каждого класса, а коэффициенты логистической регрессии показывают, насколько каждое слово увеличивает или уменьшает логарифмические шансы класса.

---

## Вопросы для самопроверки

### Для начинающих
1. Как в гауссовском наивном байесовском классификаторе вычисляются средние и дисперсии для каждого класса?
2. Зачем в мультиномиальном наивном байесовском классификаторе используется сглаживание Лапласа (параметр $\alpha$)? К чему приводит $\alpha=0$?
3. Почему наивный байесовский классификатор особенно хорошо подходит для классификации текстов? Какие свойства текстовых данных играют ему на руку?
4. Чем наивный байесовский классификатор принципиально отличается от логистической регрессии с точки зрения моделирования данных?

### Для профессионалов
1. Докажите, что если признаки действительно условно независимы при заданном классе, то наивный байесовский классификатор является оптимальным (совпадает с байесовским решающим правилом).
2. Объясните, почему при сильной корреляции признаков наивный байесовский классификатор может существенно уступать логистической регрессии. Приведите пример такой ситуации.
3. В наивном байесовском классификаторе мы игнорируем нормировочную константу $P(\mathbf{x})$, так как она не зависит от класса. В каком случае это может привести к некорректной оценке вероятностей (хотя и не меняет решающее правило)? Как это влияет на калибровку модели?

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
```


## Часть III. Байесовская линейная регрессия

В классической линейной регрессии мы точечно оценивали веса $\mathbf{w}$ методом наименьших квадратов или его регуляризованными вариантами. Байесовский подход позволяет перейти от одной «лучшей» прямой к целому распределению прямых, учитывая неопределённость в весах. Это даёт не только прогноз, но и честную оценку его надёжности.

### 1. Вероятностная модель линейной регрессии

Рассмотрим модель, в которой целевая переменная $t$ связана с вектором признаков $\boldsymbol{\phi}(\mathbf{x}) \in \mathbb{R}^M$ линейной зависимостью с аддитивным гауссовским шумом:
$$
t = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}) + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, \sigma^2).
\tag{1.1}
$$
Здесь $\sigma^2$ — дисперсия шума, которую в этом разделе будем считать известной (вопрос её оценивания отложим). Для набора из $n$ наблюдений $\{\mathbf{x}_i, t_i\}_{i=1}^n$ введём матрицу плана $\Phi \in \mathbb{R}^{n \times M}$, строки которой есть $\boldsymbol{\phi}(\mathbf{x}_i)^T$, и вектор целей $\mathbf{t} = (t_1,\dots,t_n)^T$. Функция правдоподобия для весов принимает вид многомерного нормального распределения:
$$
p(\mathbf{t} \mid \Phi, \mathbf{w}) = \mathcal{N}(\mathbf{t} \mid \Phi \mathbf{w}, \sigma^2 I).
\tag{1.2}
$$

### 2. Априорное распределение и сопряжённость

Чтобы включить априорные представления о весах, выберем сопряжённое априорное распределение — также нормальное. Удобно взять изотропный гауссиан с нулевым средним:
$$
p(\mathbf{w}) = \mathcal{N}(\mathbf{w} \mid \mathbf{0}, \tau^2 I).
\tag{2.1}
$$
Нулевое среднее означает, что до наблюдения данных мы не ожидаем, что какие‑то признаки априори важнее других (это легко обобщается). Дисперсия $\tau^2$ контролирует, насколько сильно мы верим в близость весов к нулю: малые $\tau^2$ соответствуют сильной регуляризации, большие — почти плоскому априорному распределению.

**Сопряжённость** в данном контексте означает, что априорное и апостериорное распределения принадлежат одному параметрическому семейству (здесь — нормальному). Это свойство делает байесовское обновление замкнутым и удобным для аналитических вычислений.

### 3. Вывод апостериорного распределения

Апостериорное распределение весов по теореме Байеса пропорционально произведению правдоподобия и априора:
$$
p(\mathbf{w} \mid \Phi, \mathbf{t}) \propto p(\mathbf{t} \mid \Phi, \mathbf{w}) \, p(\mathbf{w}).
$$
Логарифм апостериорной плотности с точностью до аддитивной константы равен
$$
\log p(\mathbf{w} \mid \Phi, \mathbf{t}) \propto
-\frac{1}{2\sigma^2} \|\mathbf{t} - \Phi \mathbf{w}\|^2
- \frac{1}{2\tau^2} \|\mathbf{w}\|^2.
\tag{3.1}
$$
Раскроем квадратичные формы и сгруппируем слагаемые, содержащие $\mathbf{w}$:
$$
\begin{aligned}
\log p(\mathbf{w} \mid \Phi, \mathbf{t})
&\propto -\frac{1}{2\sigma^2} \bigl( \mathbf{t}^T\mathbf{t} - 2\mathbf{t}^T\Phi\mathbf{w} + \mathbf{w}^T\Phi^T\Phi\mathbf{w} \bigr)
- \frac{1}{2\tau^2} \mathbf{w}^T\mathbf{w} \\
&\propto -\frac{1}{2} \mathbf{w}^T \left( \frac{1}{\sigma^2}\Phi^T\Phi + \frac{1}{\tau^2}I \right) \mathbf{w}
+ \frac{1}{\sigma^2} \mathbf{w}^T \Phi^T \mathbf{t}.
\end{aligned}
$$
Сравнивая с логарифмом плотности многомерного нормального распределения $\mathcal{N}(\mathbf{w} \mid \mathbf{m}_N, \mathbf{S}_N)$,
$$
\log \mathcal{N}(\mathbf{w} \mid \mathbf{m}_N, \mathbf{S}_N) = -\frac{1}{2} (\mathbf{w} - \mathbf{m}_N)^T \mathbf{S}_N^{-1} (\mathbf{w} - \mathbf{m}_N) + \text{const},
$$
идентифицируем матрицу точности (обратную ковариационной) и среднее:
$$
\boxed{\begin{aligned}
\mathbf{S}_N^{-1} &= \frac{1}{\sigma^2} \Phi^T \Phi + \frac{1}{\tau^2} I, \\
\mathbf{m}_N &= \frac{1}{\sigma^2} \mathbf{S}_N \Phi^T \mathbf{t}.
\end{aligned}}
\tag{3.2}
$$
Таким образом, апостериорное распределение весов вновь является нормальным:
$$
p(\mathbf{w} \mid \Phi, \mathbf{t}) = \mathcal{N}(\mathbf{w} \mid \mathbf{m}_N, \mathbf{S}_N).
$$

> **Углублённый взгляд: доказательство сопряжённости через дополнение квадрата**
>
> Покажем, что произведение двух многомерных нормальных плотностей даёт (с точностью до нормировки) нормальную плотность. Пусть
> $$
> p_1(\mathbf{w}) = \mathcal{N}(\mathbf{w} \mid \boldsymbol{\mu}_1, \Sigma_1), \quad
> p_2(\mathbf{w}) = \mathcal{N}(\mathbf{w} \mid \boldsymbol{\mu}_2, \Sigma_2).
> $$
> Логарифм произведения с точностью до константы:
> $$
> -\frac{1}{2} (\mathbf{w} - \boldsymbol{\mu}_1)^T \Sigma_1^{-1} (\mathbf{w} - \boldsymbol{\mu}_1)
> - \frac{1}{2} (\mathbf{w} - \boldsymbol{\mu}_2)^T \Sigma_2^{-1} (\mathbf{w} - \boldsymbol{\mu}_2) \\
> = -\frac{1}{2} \mathbf{w}^T (\Sigma_1^{-1} + \Sigma_2^{-1}) \mathbf{w}
> + \mathbf{w}^T (\Sigma_1^{-1}\boldsymbol{\mu}_1 + \Sigma_2^{-1}\boldsymbol{\mu}_2) + \text{const}.
> $$
> Это квадратичная форма, соответствующая нормальному распределению с ковариационной матрицей $\Sigma = (\Sigma_1^{-1} + \Sigma_2^{-1})^{-1}$ и средним $\boldsymbol{\mu} = \Sigma (\Sigma_1^{-1}\boldsymbol{\mu}_1 + \Sigma_2^{-1}\boldsymbol{\mu}_2)$. В нашем случае $\Sigma_1^{-1} = \frac{1}{\sigma^2}\Phi^T\Phi$, $\boldsymbol{\mu}_1 = (\Phi^T\Phi)^{-1}\Phi^T\mathbf{t}$ (MLE), $\Sigma_2^{-1} = \frac{1}{\tau^2}I$, $\boldsymbol{\mu}_2 = \mathbf{0}$. Подстановка даёт в точности (3.2).

### 4. Связь с гребневой регрессией (Ridge)

Максимум апостериорной плотности (MAP‑оценка) достигается в точке $\mathbf{m}_N$, так как для нормального распределения мода совпадает со средним. Таким образом,
$$
\hat{\mathbf{w}}_{\text{MAP}} = \mathbf{m}_N = \left( \Phi^T \Phi + \frac{\sigma^2}{\tau^2} I \right)^{-1} \Phi^T \mathbf{t}.
$$
Сравните с аналитическим решением гребневой регрессии (Ridge):
$$
\hat{\mathbf{w}}_{\text{Ridge}} = \arg\min_{\mathbf{w}} \bigl\{ \|\mathbf{t} - \Phi\mathbf{w}\|^2 + \lambda \|\mathbf{w}\|^2 \bigr\}
= \left( \Phi^T \Phi + \lambda I \right)^{-1} \Phi^T \mathbf{t}.
$$
Совпадение будет точным, если положить $\lambda = \frac{\sigma^2}{\tau^2}$. Иными словами, гребневая регрессия — это байесовская оценка с гауссовским априорным распределением на веса, а параметр регуляризации $\lambda$ имеет прямую вероятностную интерпретацию как отношение шума к априорной дисперсии.

### 5. Предсказательное распределение

Истинная сила байесовского подхода раскрывается при построении предсказательного распределения для новой точки $\mathbf{x}_*$. Вместо того чтобы подставлять точечную оценку весов, мы усредняем все возможные модели, взвешивая их апостериорной вероятностью:
$$
p(t_* \mid \mathbf{x}_*, \Phi, \mathbf{t}) = \int p(t_* \mid \mathbf{x}_*, \mathbf{w}) \, p(\mathbf{w} \mid \Phi, \mathbf{t}) \, d\mathbf{w}.
\tag{5.1}
$$
Оба сомножителя под интегралом — нормальные плотности: первая есть $\mathcal{N}(t_* \mid \mathbf{w}^T\boldsymbol{\phi}_*, \sigma^2)$, вторая — $\mathcal{N}(\mathbf{w} \mid \mathbf{m}_N, \mathbf{S}_N)$. Интеграл свёртки двух гауссиан снова даёт гауссиан. Вычислим его параметры.

Запишем $t_* = \mathbf{w}^T\boldsymbol{\phi}_* + \varepsilon_*$, где $\varepsilon_* \sim \mathcal{N}(0,\sigma^2)$ не зависит от $\mathbf{w}$. Поскольку $\mathbf{w} \sim \mathcal{N}(\mathbf{m}_N, \mathbf{S}_N)$, линейная комбинация $\mathbf{w}^T\boldsymbol{\phi}_*$ распределена как $\mathcal{N}(\mathbf{m}_N^T\boldsymbol{\phi}_*, \boldsymbol{\phi}_*^T \mathbf{S}_N \boldsymbol{\phi}_*)$. Добавление независимого шума $\varepsilon_*$ увеличивает дисперсию на $\sigma^2$. Таким образом,
$$
\boxed{p(t_* \mid \mathbf{x}_*, \Phi, \mathbf{t}) = \mathcal{N}\bigl( t_* \mid \underbrace{\mathbf{m}_N^T \boldsymbol{\phi}_*}_{\text{среднее}},\; \underbrace{\sigma^2 + \boldsymbol{\phi}_*^T \mathbf{S}_N \boldsymbol{\phi}_*}_{\text{дисперсия}} \bigr).}
\tag{5.2}
$$
Дисперсия предсказания состоит из двух частей: неизбежного шума $\sigma^2$ и вклада неопределённости в весах $\boldsymbol{\phi}_*^T \mathbf{S}_N \boldsymbol{\phi}_*$. Последний член растёт по мере удаления $\boldsymbol{\phi}_*$ от области обучающих данных, что честно отражает падение уверенности модели в незнакомых регионах.

### 6. Реализация на Python и визуализация

Реализуем класс `BayesianLinearRegression` с нуля, используя только `numpy` и `scipy.stats`. Продемонстрируем его на синтетических данных, сравнив с гребневой регрессией.

```python
class BayesianLinearRegression:
    def __init__(self, sigma2: float, tau2: float):
        self.sigma2 = sigma2   # дисперсия шума
        self.tau2 = tau2       # априорная дисперсия весов

    def fit(self, Phi: np.ndarray, t: np.ndarray):
        n, M = Phi.shape
        # Апостериорная точность и ковариация
        self.S_inv = (1/self.sigma2) * Phi.T @ Phi + (1/self.tau2) * np.eye(M)
        self.S = np.linalg.inv(self.S_inv)
        # Апостериорное среднее
        self.m = (1/self.sigma2) * self.S @ Phi.T @ t
        self.Phi_train = Phi
        return self

    def predict(self, Phi_test: np.ndarray):
        # Среднее предсказания
        mean = Phi_test @ self.m
        # Дисперсия предсказания: sigma2 + diag(Phi_test * S * Phi_test^T)
        var = self.sigma2 + np.sum(Phi_test @ self.S * Phi_test, axis=1)
        return mean, var
```

Сгенерируем синтетическую выборку $t = 3x - 2 + \varepsilon$, $x \sim U(-3,3)$, $\varepsilon \sim \mathcal{N}(0,1)$. В качестве базисных функций возьмём полиномы до 5‑й степени (включая константу). Выберем гиперпараметры $\sigma^2=1$, $\tau^2=0.5$ (умеренная регуляризация).

```python
# Генерация данных
np.random.seed(0)
n = 30
x = np.random.uniform(-3, 3, n)
t = 3 * x - 2 + np.random.randn(n)   # sigma = 1
x_test = np.linspace(-3.5, 3.5, 200)

# Полиномиальные признаки до 5-й степени
poly = PolynomialFeatures(degree=5, include_bias=True)
Phi_train = poly.fit_transform(x.reshape(-1, 1))
Phi_test = poly.transform(x_test.reshape(-1, 1))

# Байесовская регрессия
sigma2, tau2 = 1.0, 0.5
blr = BayesianLinearRegression(sigma2, tau2)
blr.fit(Phi_train, t)
mean, var = blr.predict(Phi_test)
std = np.sqrt(var)

# Гребневая регрессия с эквивалентным lambda = sigma2/tau2
ridge = make_pipeline(PolynomialFeatures(5, include_bias=True),
                      Ridge(alpha=sigma2/tau2, fit_intercept=False))
ridge.fit(x.reshape(-1,1), t)
ridge_pred = ridge.predict(x_test.reshape(-1,1))

# Визуализация
plt.figure(figsize=(10, 6))
plt.scatter(x, t, c='black', s=40, label='Обучающие данные')
plt.plot(x_test, mean, 'r-', lw=2, label='Байесовское среднее')
plt.fill_between(x_test, mean - 2*std, mean + 2*std,
                 color='gray', alpha=0.3, label='±2 стд. откл.')
# Несколько семплов из апостериорного распределения весов
for _ in range(5):
    w_sample = multivariate_normal.rvs(mean=blr.m, cov=blr.S)
    y_sample = Phi_test @ w_sample
    plt.plot(x_test, y_sample, 'b--', lw=0.8)
plt.plot(x_test, ridge_pred, 'g--', lw=1.5, label='Ridge (точечно)')
plt.xlabel('x')
plt.ylabel('t')
plt.legend()
plt.title('Байесовская линейная регрессия (полиномы степени 5)')
plt.show()
```

На графике видно, что среднее предсказание (красная линия) в точности совпадает с прогнозом гребневой регрессии (зелёная пунктирная линия). Семплы из апостериорного распределения весов (синие пунктирные кривые) демонстрируют разброс моделей, который увеличивается на краях — там, где нет данных. Серая полоса $\pm 2$ стандартных отклонения честно отражает суммарную неопределённость: она узка в середине и расширяется к границам, предупреждая нас о том, что экстраполяции доверять не стоит.

---

## Вопросы для самопроверки

### Для начинающих
1. Что даёт априорное распределение на веса в байесовской линейной регрессии? Какой эффект оказывает параметр $\tau^2$?
2. К какому семейству принадлежит апостериорное распределение весов в рассмотренной модели? Почему это удобно?
3. Что такое предсказательное распределение и чем оно отличается от точечного прогноза (одного числа)?
4. Из каких двух компонент складывается дисперсия предсказания в байесовской линейной регрессии?
5. Как параметр $\tau^2$ связан с коэффициентом регуляризации $\lambda$ в гребневой регрессии?

### Для профессионалов
1. Докажите, что гауссовское априорное распределение является сопряжённым к гауссовскому правдоподобию. Выведите формулы для апостериорных параметров $\mathbf{m}_N$ и $\mathbf{S}_N$ через дополнение квадрата.
2. Покажите, используя свойства нормальных распределений, что предсказательная дисперсия растёт по мере удаления $\boldsymbol{\phi}_*$ от центра масс обучающих данных.
3. Объясните, почему MAP‑оценка (максимум апостериорной плотности) в точности совпадает с аналитическим решением гребневой регрессии при подходящем выборе параметра регуляризации.
4. Как изменится апостериорное и предсказательное распределение, если взять априорное распределение с ненулевым средним $\mathbf{w} \sim \mathcal{N}(\mathbf{m}_0, \tau^2 I)$? Выведите соответствующие формулы.

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
```


## Часть IV. Байесовская линейная регрессия: оценивание гиперпараметров и полный байесовский вывод

В предыдущей части мы предполагали, что дисперсия шума $\sigma^2$ и априорная дисперсия весов $\tau^2$ известны. На практике эти гиперпараметры почти никогда не заданы априори, и их необходимо определять по данным. Байесовский подход предлагает изящное решение — максимизацию маргинального правдоподобия (evidence), позволяющую автоматически выбрать оптимальный баланс между сложностью модели и её соответствием данным.

### 1. Необходимость оценивания гиперпараметров

В байесовской линейной регрессии гиперпараметры $\sigma^2$ и $\tau^2$ (или эквивалентно $\lambda = \sigma^2/\tau^2$) определяют форму апостериорного распределения весов и ширину предсказательных интервалов. Слишком маленькое $\tau^2$ (сильная регуляризация) приводит к недообучению, слишком большое — к переобучению. Можно подбирать их кросс‑валидацией, но она требует разбиения данных, уменьшает обучающую выборку и даёт лишь точечную оценку без вероятностной интерпретации. Байесовская альтернатива — найти значения гиперпараметров, максимизирующие вероятность наблюдения данных, проинтегрированную по всем возможным весам. Такой подход называется **evidence approximation** (или empirical Bayes, максимизация маргинального правдоподобия типа II).

### 2. Маргинальное правдоподобие (evidence)

Маргинальное правдоподобие (evidence) определяется как интеграл от совместного распределения данных и весов по всем возможным значениям $\mathbf{w}$:
$$
p(\mathbf{t} \mid \Phi, \sigma^2, \tau^2) = \int p(\mathbf{t} \mid \Phi, \mathbf{w}, \sigma^2) \, p(\mathbf{w} \mid \tau^2) \, d\mathbf{w}.
\tag{2.1}
$$
Поскольку оба распределения — гауссовские, интеграл берётся аналитически. Заметим, что модель может быть записана как
$$
\mathbf{t} = \Phi \mathbf{w} + \boldsymbol{\varepsilon}, \qquad \mathbf{w} \sim \mathcal{N}(\mathbf{0}, \tau^2 I), \quad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \sigma^2 I),
$$
причём $\mathbf{w}$ и $\boldsymbol{\varepsilon}$ независимы. Линейная комбинация независимых гауссовских векторов распределена нормально с нулевым средним и ковариационной матрицей
$$
\mathbf{C} = \mathrm{Cov}(\mathbf{t}) = \Phi (\tau^2 I) \Phi^T + \sigma^2 I = \sigma^2 I + \tau^2 \Phi \Phi^T.
\tag{2.2}
$$
Следовательно,
$$
p(\mathbf{t} \mid \Phi, \sigma^2, \tau^2) = \mathcal{N}(\mathbf{t} \mid \mathbf{0}, \sigma^2 I + \tau^2 \Phi \Phi^T).
\tag{2.3}
$$
Логарифм маргинального правдоподобия имеет вид
$$
\log p(\mathbf{t} \mid \sigma^2, \tau^2) = -\frac{n}{2}\log(2\pi) - \frac{1}{2}\log|\mathbf{C}| - \frac{1}{2} \mathbf{t}^T \mathbf{C}^{-1} \mathbf{t}.
\tag{2.4}
$$
Эта величина интерпретируется как мера того, насколько хорошо модель с данными гиперпараметрами объясняет наблюдения, автоматически штрафуя за излишнюю сложность (через определитель $\mathbf{C}$). Максимизация evidence естественно реализует принцип «бритвы Оккама»: сложные модели, способные породить много разных наборов данных, «размазывают» evidence по большому объёму, и их маргинальное правдоподобие оказывается ниже, если только они не объясняют данные значительно лучше простых.

### 3. Оптимизация evidence

Прямая максимизация (2.4) по $\sigma^2$ и $\tau^2$ является задачей нелинейной оптимизации. Для численной устойчивости и обеспечения положительности параметров удобно работать с логарифмической параметризацией: пусть $\theta_1 = \log \sigma^2$, $\theta_2 = \log \tau^2$. Тогда $\sigma^2 = e^{\theta_1}$, $\tau^2 = e^{\theta_2}$.

Альтернативный путь — итеративные формулы, приведённые в книге Бишопа (Bishop, PRML, раздел 3.5), которые обновляют $\alpha = \sigma^2/\tau^2$ и $\beta = 1/\sigma^2$ через эффективное число параметров $\gamma = \sum_i \frac{\lambda_i}{\lambda_i + \alpha}$, где $\lambda_i$ — собственные значения $\Phi^T\Phi$. Мы реализуем более прямой метод — оптимизацию логарифма evidence с помощью `scipy.optimize.minimize`.

### 4. Сравнение с кросс‑валидацией

Evidence approximation обладает рядом преимуществ: он использует всю выборку для настройки гиперпараметров, не требует многократного переобучения модели и даёт дифференцируемый критерий. Однако он опирается на корректность вероятностной модели; если модель плохо специфицирована, evidence может давать смещённые оценки. Кросс‑валидация, с другой стороны, свободна от параметрических предположений о распределении данных, но вычислительно дороже и обладает большей дисперсией при малых выборках. Сравним оба подхода на практике.

### 5. Полный байесовский вывод с гамма-приором (упоминание)

Вместо точечной оценки гиперпараметров можно пойти ещё дальше и ввести априорные распределения на точности $1/\sigma^2$ и $1/\tau^2$ (например, гамма-распределения) и проинтегрировать по ним. Тогда предсказательное распределение для $t_*$ примет форму многомерного t-распределения Стьюдента с более тяжёлыми хвостами, что лучше отражает остаточную неопределённость в гиперпараметрах. Этот подход выходит за рамки нашей реализации, но является естественным обобщением байесовской линейной регрессии.

### 6. Код для evidence approximation

Продолжим работу с синтетическими данными и полиномиальной регрессией из предыдущей части. Напишем функцию, вычисляющую логарифм evidence (с точностью до константы), и найдём оптимальные значения $\sigma^2$ и $\tau^2$, максимизирующие эту функцию.

```python
def log_evidence(theta, Phi, t):
    """Логарифм маргинального правдоподобия.
    theta = [log(sigma2), log(tau2)].
    """
    sigma2 = np.exp(theta[0])
    tau2 = np.exp(theta[1])
    n, m = Phi.shape
    C = sigma2 * np.eye(n) + tau2 * Phi @ Phi.T
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        return -np.inf
    try:
        L = np.linalg.cholesky(C)
        alpha = np.linalg.solve(L.T, np.linalg.solve(L, t))
        quad = t @ alpha
    except np.linalg.LinAlgError:
        quad = t @ np.linalg.solve(C, t)
    return -0.5 * (logdet + quad)   # без постоянного члена -n/2 log(2π)
```

Восстановим данные и матрицу плана:

```python
# Генерация данных (повторяем из предыдущей части)
np.random.seed(0)
n = 30
x = np.random.uniform(-3, 3, n)
t = 3 * x - 2 + np.random.randn(n)
x_test = np.linspace(-3.5, 3.5, 200)

# Полиномиальные признаки
poly = PolynomialFeatures(degree=5, include_bias=True)
Phi_train = poly.fit_transform(x.reshape(-1, 1))
Phi_test = poly.transform(x_test.reshape(-1, 1))
```

Теперь оптимизируем:

```python
# Начальное приближение: log(sigma2) = 0 => sigma2=1, log(tau2)=0 => tau2=1
init_theta = np.array([0.0, 0.0])
res = minimize(lambda th: -log_evidence(th, Phi_train, t), init_theta,
               method='Nelder-Mead', options={'xatol': 1e-6, 'fatol': 1e-6})
optimal_sigma2 = np.exp(res.x[0])
optimal_tau2 = np.exp(res.x[1])
print(f"Оптимальные гиперпараметры: sigma^2 = {optimal_sigma2:.4f}, tau^2 = {optimal_tau2:.4f}")
print(f"Эквивалентный lambda (Ridge) = {optimal_sigma2/optimal_tau2:.4f}")
```

Построим график логарифма evidence как функции $\log \tau^2$ при фиксированном оптимальном $\sigma^2$:

```python
tau2_range = np.logspace(-2, 2, 50)
log_evals = []
for tau2 in tau2_range:
    log_evals.append(log_evidence([np.log(optimal_sigma2), np.log(tau2)], Phi_train, t))
plt.figure(figsize=(8,4))
plt.semilogx(tau2_range, log_evals, 'b-')
plt.axvline(optimal_tau2, color='r', linestyle='--', label=f'opt tau^2={optimal_tau2:.2f}')
plt.xlabel('tau^2')
plt.ylabel('log evidence')
plt.legend()
plt.title('Зависимость логарифма evidence от tau^2')
plt.grid()
plt.show()
```

Теперь обучим байесовскую регрессию с оптимизированными гиперпараметрами и сравним её с моделью с фиксированными (например, $\sigma^2=1, \tau^2=0.5$) и с кросс‑валидацией для Ridge.

```python
# Байесовская регрессия с оптимизированными гиперпараметрами
blr_opt = BayesianLinearRegression(optimal_sigma2, optimal_tau2)
blr_opt.fit(Phi_train, t)
mean_opt, var_opt = blr_opt.predict(Phi_test)
std_opt = np.sqrt(var_opt)

# Байесовская регрессия с фиксированными параметрами (как в предыдущей части)
blr_fixed = BayesianLinearRegression(sigma2=1.0, tau2=0.5)
blr_fixed.fit(Phi_train, t)
mean_fixed, var_fixed = blr_fixed.predict(Phi_test)
std_fixed = np.sqrt(var_fixed)

# Подбор alpha для Ridge через 5-фолд CV
ridge_cv = make_pipeline(PolynomialFeatures(5, include_bias=True),
                         Ridge(fit_intercept=False))
alphas = np.logspace(-3, 3, 20)
param_grid = {'ridge__alpha': alphas}
grid = GridSearchCV(ridge_cv, param_grid, cv=5, scoring='neg_mean_squared_error')
# Требует импорта: from sklearn.model_selection import GridSearchCV
# Но GridSearchCV уже есть в namespace? В начале нет, добавим импорт.
# Так как импорты общие, в предыдущих частях их не было. Мы можем добавить здесь.
from sklearn.model_selection import GridSearchCV
grid.fit(x.reshape(-1,1), t)
best_alpha = grid.best_params_['ridge__alpha']
print(f"Лучший alpha по CV: {best_alpha:.4f} (lambda = {best_alpha})")

# Прогноз Ridge с лучшим alpha
ridge_best = make_pipeline(PolynomialFeatures(5, include_bias=True),
                           Ridge(alpha=best_alpha, fit_intercept=False))
ridge_best.fit(x.reshape(-1,1), t)
ridge_pred_cv = ridge_best.predict(x_test.reshape(-1,1))
```

Сравним предсказания и ошибки на тестовой сетке. Так как у нас нет отдельного тестового набора, оценим качество на плотной сетке, сравнивая с истинной функцией $f(x)=3x-2$:

```python
f_true = 3 * x_test - 2
mse_opt = mean_squared_error(f_true, mean_opt)
mse_fixed = mean_squared_error(f_true, mean_fixed)
mse_cv = mean_squared_error(f_true, ridge_pred_cv)
print(f"MSE относительно истинной функции:")
print(f"  Bayesian opt:   {mse_opt:.4f}")
print(f"  Bayesian fixed: {mse_fixed:.4f}")
print(f"  Ridge CV:       {mse_cv:.4f}")

# Визуализация сравнения предсказательных интервалов
plt.figure(figsize=(10,6))
plt.scatter(x, t, c='black', s=40)
plt.plot(x_test, f_true, 'k--', label='Истинная функция')
plt.plot(x_test, mean_opt, 'r-', label='Bayes opt mean')
plt.fill_between(x_test, mean_opt-2*std_opt, mean_opt+2*std_opt,
                 color='red', alpha=0.15)
plt.plot(x_test, mean_fixed, 'b-', label='Bayes fixed mean')
plt.fill_between(x_test, mean_fixed-2*std_fixed, mean_fixed+2*std_fixed,
                 color='blue', alpha=0.1)
plt.plot(x_test, ridge_pred_cv, 'g--', label='Ridge CV')
plt.legend()
plt.title('Сравнение предсказаний')
plt.show()
```

Как правило, оптимизация evidence даёт осмысленное значение $\lambda$, и предсказательная способность близка к кросс‑валидации, но без необходимости перебора и разбиений. Предсказательные интервалы автоматически масштабируются в соответствии с оценённым шумом и априорной дисперсией.

---

## Вопросы для самопроверки

### Для начинающих
1. Что такое маргинальное правдоподобие (evidence) в контексте байесовской линейной регрессии?
2. Зачем максимизировать evidence вместо того, чтобы подбирать гиперпараметры по валидационной выборке?
3. Чем отличается evidence approximation от кросс‑валидации? В каких ситуациях каждый из методов предпочтителен?
4. Почему при полном байесовском выводе (с априорными распределениями на гиперпараметры) предсказательное распределение становится t-распределением, а не гауссовским?

### Для профессионалов
1. Выведите выражение для ковариационной матрицы $\mathbf{C} = \sigma^2 I + \tau^2 \Phi \Phi^T$ и логарифма evidence (2.4), исходя из линейной модели с гауссовским шумом и гауссовским априором.
2. Объясните, каким образом максимизация evidence реализует автоматический баланс между сложностью модели и подгонкой к данным (принцип «бритвы Оккама»). Как в выражении evidence проявляется штраф за сложность?
3. Сравните evidence approximation с EM‑алгоритмом для оценивания гиперпараметров. В чём их сходство и различие?
4. Как обобщить вывод evidence на случай, когда $\sigma^2$ неизвестна и на неё также вводится гамма-априорное распределение? Каким станет предсказательное распределение?



```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit  # сигмоида
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
```


## Часть V. Байесовская логистическая регрессия

Логистическая регрессия — основной инструмент бинарной классификации. В байесовской трактовке мы не ищем одну «оптимальную» разделяющую гиперплоскость, а рассматриваем целое распределение весов, что позволяет честно оценивать неопределённость предсказаний. Однако, в отличие от линейной регрессии, здесь апостериорное распределение не является гауссовским, и нам потребуются приближённые методы.

### 1. Постановка задачи и проблема отсутствия сопряжённости

В логистической регрессии вероятность принадлежности к классу $y=1$ задаётся сигмоидной функцией:
$$
p(y=1 \mid \mathbf{x}, \mathbf{w}) = \sigma(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x})), \qquad \sigma(z) = \frac{1}{1+e^{-z}}.
$$
Для обучающей выборки $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$ с $y_i \in \{0,1\}$ правдоподобие имеет вид произведения бернуллиевских вероятностей:
$$
p(\mathbf{y} \mid \Phi, \mathbf{w}) = \prod_{i=1}^n \sigma_i^{y_i} (1-\sigma_i)^{1-y_i}, \qquad \sigma_i = \sigma(\mathbf{w}^T\boldsymbol{\phi}(\mathbf{x}_i)).
$$
Как и ранее, мы выбираем гауссовское априорное распределение на веса:
$$
p(\mathbf{w}) = \mathcal{N}(\mathbf{w} \mid \mathbf{0}, \tau^2 I).
$$
Апостериорное распределение по теореме Байеса
$$
p(\mathbf{w} \mid \mathcal{D}) \propto p(\mathbf{y} \mid \Phi, \mathbf{w})\,p(\mathbf{w})
$$
уже не является гауссовским: сигмоида в правдоподобии разрушает квадратичную структуру. Аналитическое вычисление апостериорных моментов и предсказательного распределения становится невозможным. Нам нужен метод приближённого вывода — один из самых простых и популярных — **лапласовская аппроксимация**.

### 2. Лапласовская аппроксимация

Идея лапласовской аппроксимации состоит в том, чтобы приблизить логарифм апостериорной плотности квадратичной формой в окрестности его максимума. Это даёт гауссовское аппроксимирующее распределение.

**Шаг 1: нахождение моды (MAP-оценки).**  
Мода апостериорного распределения $\mathbf{w}_{\text{MAP}}$ минимизирует отрицательный логарифм апостериора (с точностью до константы):
$$
J(\mathbf{w}) = -\sum_{i=1}^n \bigl[y_i \log \sigma_i + (1-y_i)\log(1-\sigma_i)\bigr] + \frac{1}{2\tau^2}\|\mathbf{w}\|^2.
$$
Это задача выпуклой оптимизации (сигмоида логарифмически вогнута), которая решается, например, методом BFGS.

**Шаг 2: вычисление гессиана.**  
Разложим логарифм апостериора в ряд Тейлора до второго порядка вокруг $\mathbf{w}_{\text{MAP}}$:
$$
\log p(\mathbf{w} \mid \mathcal{D}) \approx \log p(\mathbf{w}_{\text{MAP}} \mid \mathcal{D}) - \frac{1}{2} (\mathbf{w} - \mathbf{w}_{\text{MAP}})^T \mathbf{H} (\mathbf{w} - \mathbf{w}_{\text{MAP}}),
$$
где $\mathbf{H}$ — отрицательный гессиан логарифма апостериора в точке максимума. Для нашей модели он вычисляется явно:
$$
\mathbf{H} = -\nabla\nabla \log p(\mathbf{w} \mid \mathcal{D}) = \Phi^T \mathbf{D} \Phi + \frac{1}{\tau^2} I,
$$
$$
\mathbf{D} = \operatorname{diag}\bigl(\sigma_i(1-\sigma_i)\bigr), \qquad \sigma_i = \sigma(\mathbf{w}_{\text{MAP}}^T \boldsymbol{\phi}(\mathbf{x}_i)).
$$
Таким образом, лапласовская аппроксимация даёт
$$
q(\mathbf{w}) = \mathcal{N}(\mathbf{w} \mid \mathbf{w}_{\text{MAP}}, \mathbf{S}_N), \qquad \mathbf{S}_N = \mathbf{H}^{-1}.
$$

### 3. Предсказательное распределение

Имея аппроксимирующее распределение $q(\mathbf{w})$, мы хотим вычислить вероятность для нового объекта $\mathbf{x}_*$:
$$
p(y_*=1 \mid \mathbf{x}_*, \mathcal{D}) \approx \int \sigma(\mathbf{w}^T\boldsymbol{\phi}_*) \, q(\mathbf{w}) \, d\mathbf{w}.
$$
Этот одномерный интеграл не берётся аналитически, но его можно приблизить. Один из способов — **приближение МакКаллаха** (MacKay, 1992), основанное на близости логистической функции и пробит-функции. Свёртка логистической сигмоиды с гауссовой плотностью хорошо аппроксимируется сигмоидой от масштабированного среднего:
$$
\int \sigma(a) \, \mathcal{N}(a \mid \mu, s^2) \, da \approx \sigma\!\left( \kappa(s^2)\,\mu \right), \qquad
\kappa(s^2) = \left(1 + \frac{\pi s^2}{8}\right)^{-1/2}.
$$
В нашем случае $a = \mathbf{w}^T\boldsymbol{\phi}_*$ при фиксированном $\boldsymbol{\phi}_*$ имеет распределение $\mathcal{N}(\mu_*, s_*^2)$, где
$$
\mu_* = \mathbf{w}_{\text{MAP}}^T \boldsymbol{\phi}_*, \qquad s_*^2 = \boldsymbol{\phi}_*^T \mathbf{S}_N \boldsymbol{\phi}_*.
$$
Следовательно,
$$
p(y_*=1 \mid \mathbf{x}_*, \mathcal{D}) \approx \sigma\!\left( \frac{\mu_*}{\sqrt{1 + \frac{\pi}{8} s_*^2}} \right).
\tag{3.1}
$$
Эта формула учитывает неопределённость в весах: чем больше $s_*^2$, тем ближе предсказанная вероятность к $0.5$, что отражает неуверенность модели.

Альтернативный, более точный, но затратный способ — **сэмплирование Монте‑Карло**. Мы генерируем $S$ весов из $q(\mathbf{w})$, для каждого вычисляем сигмоиду и усредняем:
$$
p(y_*=1) \approx \frac{1}{S} \sum_{s=1}^S \sigma\bigl((\mathbf{w}^{(s)})^T \boldsymbol{\phi}_*\bigr), \quad \mathbf{w}^{(s)} \sim \mathcal{N}(\mathbf{w}_{\text{MAP}}, \mathbf{S}_N).
$$

> **Углублённый взгляд: происхождение коэффициента $\kappa$**
>
> Логистическая функция $\sigma(x)$ и пробит-функция $\Phi(x)$ (функция распределения стандартного нормального закона) связаны приближённым равенством $\sigma(x) \approx \Phi(\lambda x)$ с $\lambda = \sqrt{\pi/8}$. Тогда свёртка с гауссовой плотностью $\mathcal{N}(\mu,s^2)$:
> $$
> \int \sigma(a)\,\mathcal{N}(a|\mu,s^2)\,da \approx \int \Phi(\lambda a)\,\mathcal{N}(a|\mu,s^2)\,da
> = \Phi\!\left(\frac{\lambda\mu}{\sqrt{1+\lambda^2 s^2}}\right),
> $$
> поскольку свёртка пробит-функции с гауссианом снова даёт пробит-функцию. Возвращаясь к сигмоиде, получаем $\sigma\bigl(\mu / \sqrt{1 + \pi s^2/8}\bigr)$.

### 4. Реализация на Python

Создадим класс `BayesianLogisticRegressionLaplace`, следуя изложенной теории.

```python
class BayesianLogisticRegressionLaplace:
    def __init__(self, tau2=1.0):
        self.tau2 = tau2

    def fit(self, X, y):
        n, d = X.shape
        # Добавляем константный признак
        Phi = np.hstack([np.ones((n, 1)), X])
        self.Phi_ = Phi
        self.d_ = d + 1

        # Целевая функция (отрицательный логарифм апостериора)
        def loss(w):
            z = Phi @ w
            sigma = expit(z)
            # Регуляризация: -log p(w) с точностью до константы
            log_prior = -0.5 * np.sum(w**2) / self.tau2
            # Логарифм правдоподобия
            log_lik = np.sum(y * np.log(sigma + 1e-15) + (1-y)*np.log(1 - sigma + 1e-15))
            return -(log_lik + log_prior)

        def grad(w):
            z = Phi @ w
            sigma = expit(z)
            return Phi.T @ (sigma - y) + w / self.tau2

        # Начальное приближение
        w0 = np.zeros(self.d_)
        res = minimize(loss, w0, jac=grad, method='BFGS')
        self.w_map = res.x
        # Вычисляем гессиан и ковариацию
        sigma_map = expit(Phi @ self.w_map)
        D = np.diag(sigma_map * (1 - sigma_map))
        H = Phi.T @ D @ Phi + np.eye(self.d_) / self.tau2
        self.S = np.linalg.inv(H)

    def predict_proba(self, X, method='probit'):
        n, d = X.shape
        Phi = np.hstack([np.ones((n, 1)), X])
        mu = Phi @ self.w_map
        if method == 'probit':
            var = np.sum(Phi @ self.S * Phi, axis=1)
            kappa = 1.0 / np.sqrt(1 + np.pi * var / 8)
            proba = expit(mu * kappa)
            return proba
        elif method == 'monte_carlo':
            S = 1000
            w_samples = np.random.multivariate_normal(self.w_map, self.S, size=S)
            proba = np.mean(expit(Phi @ w_samples.T), axis=1)
            return proba
        else:
            raise ValueError("method must be 'probit' or 'monte_carlo'")

    def predict(self, X, method='probit', threshold=0.5):
        proba = self.predict_proba(X, method=method)
        return (proba >= threshold).astype(int)
```

Протестируем на breast_cancer и сравним с обычной логистической регрессией из `sklearn`.

```python
# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Наша модель с tau2=10 (слабая регуляризация)
blr = BayesianLogisticRegressionLaplace(tau2=10.0)
blr.fit(X_train, y_train)
# Предсказание двумя методами
prob_probit = blr.predict_proba(X_test, method='probit')
prob_mc = blr.predict_proba(X_test, method='monte_carlo')
acc_probit = accuracy_score(y_test, prob_probit >= 0.5)
acc_mc = accuracy_score(y_test, prob_mc >= 0.5)
print(f"Bayesian LR (probit approx): accuracy={acc_probit:.4f}")
print(f"Bayesian LR (MC approx):     accuracy={acc_mc:.4f}")

# Обычная логистическая регрессия (C = 1/lambda ~ большое, почти MLE)
lr = LogisticRegression(C=1e10, max_iter=1000)
lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test))
print(f"LogisticRegression (MLE):    accuracy={lr_acc:.4f}")
```

При большом $\tau^2$ MAP-оценка близка к MLE, и точность должна быть схожей. Уменьшение $\tau^2$ приводит к сжатию коэффициентов.

Для визуализации неопределённости возьмём один признак (например, `mean radius`) и обучим модель на нём. Построим график предсказанной вероятности с доверительным интервалом.

```python
# Одномерный пример: используем признак mean radius (индекс 0)
X_single = X_train[:, 0].reshape(-1, 1)
X_test_single = X_test[:, 0].reshape(-1, 1)

blr1d = BayesianLogisticRegressionLaplace(tau2=1.0)
blr1d.fit(X_single, y_train)

# Сетка для визуализации
x_plot = np.linspace(X_single.min(), X_single.max(), 200).reshape(-1, 1)
proba_plot = blr1d.predict_proba(x_plot, method='probit')

# Сэмплируем веса для визуализации неопределённости
w_samples = np.random.multivariate_normal(blr1d.w_map, blr1d.S, size=100)
Phi_plot = np.hstack([np.ones((200,1)), x_plot])
curves = expit(Phi_plot @ w_samples.T)

plt.figure(figsize=(8,5))
plt.scatter(X_single, y_train, alpha=0.3, label='train')
plt.plot(x_plot, proba_plot, 'r-', lw=2, label='predicted probability')
# Отображаем несколько семплов
for i in range(20):
    plt.plot(x_plot, curves[:, i], 'b-', alpha=0.1)
plt.xlabel('mean radius (standardized)')
plt.ylabel('P(malignant)')
plt.legend()
plt.title('Bayesian Logistic Regression (1D)')
plt.show()
```

На графике видно, что в области плотного расположения точек разброс кривых мал, а на краях, где данных нет, модели сильно расходятся — честный признак неуверенности.

---

## Вопросы для самопроверки

### Для начинающих
1. Почему в байесовской логистической регрессии апостериорное распределение весов не является гауссовским и не может быть найдено аналитически?
2. Что такое лапласовская аппроксимация и в чём её основная идея?
3. Чем предсказание с учётом неопределённости весов (через интегрирование) отличается от точечного прогноза по MAP-оценке?
4. Как параметр $\tau^2$ влияет на степень сжатия коэффициентов в байесовской логистической регрессии?

### Для профессионалов
1. Выведите явное выражение для гессиана логарифма апостериора в логистической регрессии. Почему он всегда положительно определён?
2. Докажите приближение пробит-функцией: покажите, что $\int \sigma(a)\,\mathcal{N}(a|\mu,s^2)\,da \approx \sigma(\mu / \sqrt{1+\pi s^2/8})$, используя связь сигмоиды и пробит-функции.
3. Обсудите недостатки лапласовской аппроксимации. В каких случаях она может давать плохое приближение (например, мультимодальный апостериор, сильная асимметрия)? Какие альтернативные методы приближённого вывода существуют?


## Часть VI. Связь L1/L2‑регуляризации с байесовским выводом и заключение

В предыдущих разделах мы двигались от простых точечных оценок к полному байесовскому выводу: научились работать с распределениями параметров, вычислять апостериорные средние и предсказательные интервалы. Теперь настало время связать эти идеи с привычными методами регуляризации и подвести итог: что нам даёт байесовский подход по сравнению с классическим машинным обучением?

### 1. MAP‑оценки и регуляризация

Вспомним общую формулу максимума апостериорной плотности (MAP):
$$
\hat{\mathbf{w}}_{\text{MAP}} = \arg\max_{\mathbf{w}} \bigl[ \log p(\mathcal{D} \mid \mathbf{w}) + \log p(\mathbf{w}) \bigr].
\tag{1.1}
$$
Первое слагаемое — логарифм правдоподобия, второе — логарифм априорной плотности. Именно выбор априорного распределения определяет вид регуляризационного штрафа.

- **Гауссовское априорное $\leftrightarrow$ $L_2$-регуляризация.**  
  Если $p(\mathbf{w}) \propto \exp\!\bigl(-\frac{1}{2\tau^2}\|\mathbf{w}\|_2^2\bigr)$, то $\log p(\mathbf{w}) = -\frac{1}{2\tau^2}\|\mathbf{w}\|_2^2 + \text{const}$. Максимизация (1.1) эквивалентна минимизации
  $$
  -\log p(\mathcal{D} \mid \mathbf{w}) + \frac{1}{2\tau^2}\|\mathbf{w}\|_2^2,
  $$
  что в точности совпадает с $L_2$-регуляризованной (ridge) задачей с параметром $\lambda = 1/(2\tau^2)$ (или $\lambda = \sigma^2/\tau^2$ в зависимости от параметризации). Гауссовское распределение «концентрирует» вероятность вокруг нуля, но не обнуляет веса полностью.

- **Лапласовское априорное $\leftrightarrow$ $L_1$-регуляризация.**  
  Если $p(\mathbf{w}) \propto \exp\!\bigl(-\frac{1}{\tau}\|\mathbf{w}\|_1\bigr)$, где $\|\mathbf{w}\|_1 = \sum_j |w_j|$, то логарифм априорной плотности пропорционален $-\|\mathbf{w}\|_1$. Тогда MAP-задача принимает вид
  $$
  \min_{\mathbf{w}} \bigl[ -\log p(\mathcal{D} \mid \mathbf{w}) + \frac{1}{\tau}\|\mathbf{w}\|_1 \bigr],
  $$
  что является в точности задачей Lasso (Least Absolute Shrinkage and Selection Operator). Параметр $\tau$ управляет силой сжатия. Лапласовское распределение имеет более острый пик в нуле, что побуждает многие координаты $\mathbf{w}$ становиться строго нулевыми.

Геометрически форму априорного распределения удобно представить на плоскости: линии равной плотности гауссовского распределения — окружности (эллипсоиды), лапласовского — ромбы. При ограничении на бюджет параметров решение ищется в точке касания контура правдоподобия и области, задаваемой априорным штрафом. Для $L_1$ точка касания часто лежит на оси координат, зануляя часть параметров; для $L_2$ касание обычно происходит во внутренней точке, сохраняя все параметры ненулевыми. Именно поэтому Lasso способствует разреженности.

```python
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, laplace

# Визуализация одномерных априорных распределений
x = np.linspace(-3, 3, 300)
tau = 1.0
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(x, norm.pdf(x, scale=tau), 'b-', label='Гауссовское (L2)')
ax[0].plot(x, laplace.pdf(x, scale=tau), 'r-', label='Лапласовское (L1)')
ax[0].set_title('Одномерные априорные плотности')
ax[0].legend()
ax[0].set_xlabel('w')
ax[0].set_ylabel('p(w)')

# Двумерная иллюстрация: линии уровня
w1, w2 = np.meshgrid(x, x)
pos = np.dstack((w1, w2))
# Гауссиан (изотропный)
gauss_2d = np.exp(-0.5 * (w1**2 + w2**2) / tau**2)
# Лаплас (независимые компоненты)
laplace_2d = np.exp(-(np.abs(w1) + np.abs(w2)) / tau)
ax[1].contour(w1, w2, gauss_2d, levels=[0.7, 0.3, 0.1], colors='blue', linestyles='-')
ax[1].contour(w1, w2, laplace_2d, levels=[0.7, 0.3, 0.1], colors='red', linestyles='--')
ax[1].set_title('Линии уровня (синий – L2, красный – L1)')
ax[1].set_xlabel('w1'); ax[1].set_ylabel('w2')
plt.suptitle('Связь априорных распределений и регуляризации')
plt.show()
```

> **Углублённый взгляд: почему лапласовское априорное порождает разреженность**
>
> Рассмотрим задачу минимизации $J(\mathbf{w}) = L(\mathbf{w}) + \lambda \|\mathbf{w}\|_1$, где $L(\mathbf{w}) = -\log p(\mathcal{D}\mid\mathbf{w})$ — выпуклая дифференцируемая функция. Условие оптимальности для координаты $w_j$ записывается с использованием субградиента модуля:
> $$
> \frac{\partial L}{\partial w_j} + \lambda \cdot \partial |w_j| \ni 0,
> $$
> где $\partial |w_j| = \{-1\}$ при $w_j<0$, $\{1\}$ при $w_j>0$ и $[-1,1]$ при $w_j=0$. Если $|\partial L/\partial w_j| < \lambda$, то условие выполняется при $w_j=0$, и координата зануляется. Для $L_2$-регуляризации производная штрафа равна $2\lambda w_j$, и условие $\partial L/\partial w_j + 2\lambda w_j = 0$ почти всегда даёт ненулевое решение. Таким образом, $L_1$ создаёт «зону нечувствительности», в которой параметры остаются равными нулю.

### 2. Сравнение точечных MAP‑оценок и полного байесовского вывода

MAP-оценка даёт лишь одну точку в пространстве параметров — моду апостериорного распределения. Она игнорирует всю остальную информацию о неопределённости: ковариацию, асимметрию, возможную мультимодальность. Полный байесовский вывод, напротив, оперирует всем апостериорным распределением и строит предсказания усреднением по нему.

Рассмотрим простой пример с двумя сильно коррелированными признаками. Lasso (MAP с лапласовским априорным) выберет один из них, полностью проигнорировав второй, в то время как апостериорное распределение покажет сильную отрицательную корреляцию между весами и широкую область правдоподобных комбинаций. Байесовское предсказание, усредняя по этой области, учтёт неопределённость и может дать более стабильный прогноз, чем жёсткий отбор переменных. Аналогично, Ridge (MAP с гауссовским априорным) даёт сжатые, но ненулевые веса; байесовский подход с тем же априорным дополнительно снабжает их доверительными интервалами.

Кроме того, полный байесовский вывод приводит к лучшей калибровке вероятностей, особенно на малых выборках. MAP-предсказание часто оказывается излишне уверенным, потому что не учитывает разброс параметров.

### 3. Evidence Approximation для логистической регрессии (кратко)

Аналогично линейной регрессии, для логистической можно максимизировать маргинальное правдоподобие (evidence) по гиперпараметрам (например, $\tau^2$). Прямое вычисление evidence $p(\mathcal{D}\mid\tau^2) = \int p(\mathcal{D}\mid\mathbf{w})p(\mathbf{w}\mid\tau^2)d\mathbf{w}$ невозможно из-за отсутствия сопряжённости. Однако можно применить лапласовскую аппроксимацию интеграла: разложить логарифм подынтегрального выражения вокруг моды $\mathbf{w}_{\text{MAP}}$ и проинтегрировать получившуюся гауссовскую аппроксимацию. Это даёт приближённый логарифм evidence:
$$
\log p(\mathcal{D}\mid\tau^2) \approx \log p(\mathcal{D}\mid\mathbf{w}_{\text{MAP}}) + \log p(\mathbf{w}_{\text{MAP}}\mid\tau^2) - \frac{1}{2}\log|\mathbf{H}| + \text{const},
$$
где $\mathbf{H}$ — гессиан из раздела о лапласовской аппроксимации. Оптимизируя эту величину по $\tau^2$ (аналитически или численно), можно автоматически настроить силу регуляризации без кросс‑валидации.

### 4. Итоговая таблица: от точечных оценок к полному байесовскому выводу

| Подход | Модель параметров | Оценка | Учёт неопределённости | Сложность |
|--------|-------------------|--------|-----------------------|-----------|
| **MLE** | Фиксированные, неизвестные | Максимизация правдоподобия | Нет (точечная) | Низкая |
| **MAP с гауссовским априором (Ridge)** | Случайные, априорное распределение | Мода апостериорного | Нет (игнорируется ковариация) | Средняя (замкнутая форма для линейной регрессии) |
| **MAP с лапласовским априором (Lasso)** | Случайные, априорное Лапласа | Мода апостериорного, разреженная | Нет | Средняя (итеративная оптимизация) |
| **Лапласовская аппроксимация** | Случайные, априорное | Аппроксимирующее нормальное распределение (центр + ковариация) | Приближённый (гауссовский) | Выше (требуется вычисление гессиана) |
| **Полный байесовский вывод (MCMC)** | Случайные, априорное | Выборка из апостериорного, гистограммы, предсказательные интервалы | Полный (произвольная форма) | Высокая (вычислительно затратен) |

**Когда что применять:**
- *Большие данные*, задача — точечный прогноз с хорошим качеством: MLE или MAP с $L_2$ (Ridge) / $L_1$ (Lasso).
- *Малые данные* или когда важна оценка неопределённости (медицина, финансы): байесовский подход с лапласовской аппроксимацией или MCMC.
- *Автоматический выбор гиперпараметров без валидационной выборки*: evidence approximation (эмпирический Байес).

### 5. Заключение по главе

Байесовская парадигма предлагает последовательный вероятностный взгляд на машинное обучение. Параметры модели перестают быть фиксированными числами и становятся случайными величинами с собственными распределениями. Это позволяет:

- **Включать априорные знания** о предметной области через выбор $p(\mathbf{w})$.
- **Автоматически получать регуляризацию**: гауссовское априорное $\rightarrow$ Ridge, лапласовское $\rightarrow$ Lasso.
- **Честно измерять неопределённость** — как в оценках параметров, так и в предсказаниях — через апостериорные распределения и предсказательные интервалы.
- **Строить полностью вероятностные модели**, в которых все неизвестные величины (включая гиперпараметры) могут быть наделены распределениями.

Мы увидели, как эти принципы реализуются в наивном байесовском классификаторе, в линейной и логистической регрессии. Лапласовская аппроксимация дала компромисс между вычислительной эффективностью и учётом неопределённости. Эти идеи естественно обобщаются на более сложные модели — байесовские нейронные сети, гауссовские процессы и другие, — где неопределённость играет ключевую роль.

---

## Вопросы для самопроверки (объединяют всю главу)

### Для начинающих
1. Что такое MAP-оценка и как она связана с максимизацией правдоподобия и априорным распределением?
2. Какое априорное распределение соответствует $L_2$-регуляризации (Ridge), а какое — $L_1$-регуляризации (Lasso)?
3. Почему байесовский подход даёт больше информации, чем одна лишь MAP-оценка?
4. Что такое evidence approximation и для чего он используется?
5. Чем отличается апостериорное распределение от точечной оценки параметров?
6. В каком случае предпочтительнее использовать полный байесовский вывод (например, MCMC), а в каком — достаточно MAP-оценки?

### Для профессионалов
1. Выведите MAP-оценку для линейной регрессии с лапласовским априорным распределением и покажите её эквивалентность Lasso. Как выбор $\tau$ влияет на разреженность?
2. Докажите, что маргинальное правдоподобие для байесовской линейной регрессии с гауссовским априором и шумом имеет вид $p(\mathbf{t} \mid \sigma^2, \tau^2) = \mathcal{N}(\mathbf{t} \mid \mathbf{0}, \sigma^2 I + \tau^2 \Phi \Phi^T)$.
3. Объясните, почему evidence approximation может быть предпочтительнее кросс‑валидации при малой выборке. Какие допущения при этом делаются?
4. Сравните смещение и дисперсию предсказаний, полученных MAP-оценкой и полным байесовским усреднением. Какое из них обычно имеет меньшую дисперсию?
5. Как лапласовское априорное распределение (через субградиент) математически приводит к разреженным решениям?

---

## Задания повышенной сложности

1. **Аналитическое маргинальное правдоподобие.** Выведите формулу для логарифма evidence байесовской линейной регрессии и реализуйте его численную оптимизацию для подбора $\sigma^2$ и $\tau^2$. Сравните выбранные параметры с оптимальными по кросс‑валидации.
2. **MCMC для логистической регрессии.** Реализуйте алгоритм Метрополиса–Гастингса для логистической регрессии с гауссовским априором. Постройте гистограммы апостериорных распределений весов и сравните их с аппроксимацией Лапласа. Оцените, насколько хорошо лапласовская аппроксимация описывает апостериорную ковариацию.
3. **Предсказательная дисперсия.** Докажите, что для байесовской линейной регрессии с известной $\sigma^2$ и гауссовским априором предсказательная дисперсия $t_*$ всегда превосходит $\sigma^2$, и покажите, что разность растёт при удалении от обучающих данных.
4. **Эксперимент со степенью полинома.** Для полиномиальной регрессии степени $1$–$10$ вычислите апостериорное распределение весов (байесовский подход с фиксированными гиперпараметрами). Для каждой степени оцените след матрицы $\mathbf{S}_N$ как меру апостериорной неопределённости. Подтвердите, что с ростом степени неопределённость возрастает.
5. **Калибровка вероятностей.** В задаче бинарной классификации с несбалансированными классами сравните калибровку предсказанных вероятностей (Brier score, reliability diagram) для MAP-логистической регрессии и байесовской логистической регрессии с лапласовской аппроксимацией. Покажите, что байесовский подход даёт лучше откалиброванные вероятности, особенно при малом размере выборки.